# Traffic Light RL Optimization — Analysis

Final analysis notebook for the Regression Analysis and Time Series course project.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from config import DEFAULT_CONFIG

## 1. Training Curves

In [ ]:
from state.redis_manager import RedisStateManager
from visualization.plots import plot_training_curves

redis_mgr = RedisStateManager()
metrics = redis_mgr.get_metrics()
print(f"Collected {len(metrics)} metric snapshots")
plot_training_curves(metrics, save_path="../figures/training_curves.png", show=True)

## 2. RL Agent vs Fixed-Cycle Baseline

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy
from agent.ppo_agent import load_model
from agent.baseline import evaluate_baseline
from environment.wrappers import make_sb3_env
from visualization.plots import plot_comparison

model = load_model('models/ppo_traffic_final')
eval_env = make_sb3_env(seed=99)

# Collect per-episode rewards for PPO
ppo_rewards = []
for ep in range(20):
    obs = eval_env.reset()
    done = False
    total_r = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = eval_env.step(action)
        total_r += reward[0]
    ppo_rewards.append(total_r)

# Baseline
baseline = evaluate_baseline(n_episodes=20)
# Approximate per-episode baseline rewards from stats
baseline_rewards = [baseline['mean_reward'] / 2] * 20  # rough approx

print(f"PPO mean: {np.mean(ppo_rewards):.2f}")
print(f"Baseline mean: {baseline['mean_reward']:.2f}")

plot_comparison(ppo_rewards, baseline_rewards, save_path='../figures/comparison.png')

## 3. Hyperparameter Optimization Results

In [ ]:
import optuna
from visualization.plots import plot_optuna_results

try:
    study = optuna.load_study(study_name='traffic_ppo', storage='sqlite:///../optuna_study.db')
    print(f"Best params: {study.best_params}")
    print(f"Best value: {study.best_value}")
    plot_optuna_results(study, save_path='../figures/optuna.png')
except Exception as e:
    print(f"No Optuna results available: {e}")

## 4. Environment Dynamics Analysis

In [ ]:
from environment.traffic_env import TrafficLightEnv

env = TrafficLightEnv()
obs, _ = env.reset(seed=42)

queues_a, queues_b, rewards_log = [], [], []

while env.agents:
    obs_a = obs['light_A']
    action, _ = model.predict(obs_a, deterministic=True)
    actions = {'light_A': int(action), 'light_B': int(action)}
    obs, rewards, _, _, infos = env.step(actions)
    info = infos.get('light_A', {})
    queues_a.append(info.get('queue_a', 0))
    queues_b.append(info.get('queue_b', 0))
    rewards_log.append(rewards.get('light_A', 0))

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(queues_a, label='Queue A', color='tab:blue')
axes[0].plot(queues_b, label='Queue B', color='tab:orange')
axes[0].set_ylabel('Queue Length')
axes[0].legend()
axes[0].set_title('Queue Dynamics Under Trained Agent')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rewards_log, color='tab:green')
axes[1].set_ylabel('Reward')
axes[1].set_xlabel('Step')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/queue_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()